In [ ]:
   from booking import Booking, HotelBooking

In [ ]:
# CSV Load 

import csv

def load_bookings(path):
    with open(path, newline ="") as f:
        reader = csv.DictReader(f)
        bookings = []
        for row in reader:
            bookings.append(HotelBooking.from_dict(row))
        return bookings    



In [ ]:
bookings = load_bookings("bookings.csv")
print(len(bookings))
print(bookings[0])
print(Booking.booking_count)
print(HotelBooking.booking_count)

b = bookings[0]
print(type(b.price),b.price)
print(type(b.start_date), b.start_date)
print("Booking class commission rate: ", Booking.commission_rate)
print("HotelBooking class commission rate: ", HotelBooking.commission_rate)

20
HotelBooking(booking_id=1001, guest_name='KukHwa Kim', hotel_name='Grand Palace Hotel', price=1250.0, status='confirmed')
40
40
<class 'float'> 1250.0
<class 'datetime.date'> 2026-10-15
Booking class commission rate:  0.05
HotelBooking class commission rate:  0.111


In [ ]:
test = {"booking_id": "T1", "guest_name": "Test", "hotel_name": "Test Hotel",
        "booking_date": "2026-09-01", "check_in": "2026-10-01", "check_out": "2026-10-03",
        "depart_country": "Canada", "destination_country": "Canada",
        "price": "100", "status":""}
print(Booking.from_dict(test).status)

   

AttributeError: type object 'Booking' has no attribute 'from_dict'

In [ ]:
# Filtering

def filter_by_status(bookings, status):
    filtered = []
    for booking in bookings:
        if booking.status == status:
            filtered.append(booking)
    return filtered

confirmed = filter_by_status(bookings, "confirmed")
print(len(confirmed))
for b in confirmed:
    print(b.booking_id, b.status)

cancelled = filter_by_status(bookings, "cancelled")
print(len(cancelled))
for b in cancelled:
    print(b.booking_id, b.status)    

In [ ]:
# list comprehension

def filter_by_status(bookings, status):
    return [b for b in bookings if b.status == status]

print(len(filter_by_status(bookings,"confirmed")))
print(len(filter_by_status(bookings,"cancelled")))
print([x.booking_id for x in filter_by_status(bookings,"confirmed")])

def filter_by_provider(bookings,provider):
    return [b for b in bookings if b.provider == provider]

print(len(filter_by_provider(bookings,"Expedia")))
print(len(filter_by_provider(bookings,"Booking.com")))


In [ ]:
# sort
def sort_by_price(bookings,descending=False):
    return sorted(bookings, key=lambda b:b.price, reverse=descending)

print([(x.booking_id,x.price) for x in sort_by_price(bookings,descending=False)[:3]]) # smallest one first
print([(x.booking_id,x.price) for x in sort_by_price(bookings,descending=True)[:3]]) # biggest one first

def sort_by_checkin(bookings,descending=False):
    return sorted(bookings, key=lambda b: b.check_in, reverse=descending) 

print([(x.booking_id, x.check_in) for x in sort_by_checkin(bookings,descending=False)[:3]]) #smallest one first
print([(x.booking_id, x.check_in) for x in sort_by_checkin(bookings,descending=True)[:3]])  #biggest one first

In [ ]:
# Sum and Average
def total_revenue(bookings):
    return sum( b.price for b in  bookings)

def average_price(bookings):
    if not bookings:
        return 0
    else:
        return round(total_revenue(bookings)/len(bookings),2)

def total_commission(bookings):
    return round(sum(b.commission() for b in bookings),2)

print(average_price(bookings))
print(average_price([]))
print(total_commission(bookings))

#print(total_commission(bookings))
#average price, safe for empty lists, rounded to 2 decimals
#total_commission(bookings): sum of each booking's commission(). Your method already returns 0 for cancelled ones, so no filtering needed.

In [ ]:
# Group by
def count_by_provider(bookings):
     counts = {}
     for b in bookings:
          if b.provider not in counts:
               counts[b.provider] = 1
          else:
               counts[b.provider] += 1 

     return counts
print(count_by_provider(bookings))


def revenue_by_provider(bookings):
    sum_revenue={}
    for b in bookings:
         if b.provider not in sum_revenue:
              sum_revenue[b.provider] = b.price 
         else:
              sum_revenue[b.provider] += b.price 
    return sum_revenue

print(revenue_by_provider(bookings))




In [ ]:
# comparing with pandas

import pandas as pd 

df = pd.read_csv("bookings.csv", parse_dates=["booking_date","check_in","check_out"])

df.head(10)

In [ ]:
# filter_by_status
print([b.booking_id for b in filter_by_status(bookings,"confirmed")])
print(df[df["status"]=="confirmed"]["booking_id"].to_list())


In [ ]:
# sort by price
print([(b.booking_id,b.price) for b in sort_by_price(bookings,descending=True)[:3]])
print(list(df.sort_values("price", ascending=False)[:3][["booking_id","price"]].itertuples(index=False,name=None)))

In [ ]:
#total revenue & average price
print("total revenue:", total_revenue(bookings))
print("average price:", average_price(bookings))

print(df["price"].sum())
print(df["price"].mean())

assert total_revenue(bookings) == df["price"].sum()

In [ ]:
# count by provider & revenue by provider
print(count_by_provider(bookings))
print(revenue_by_provider(bookings))

print(df.groupby("provider",sort=False).size().to_dict())
print(df.groupby("provider",sort=False)["price"].sum().to_dict())